In [1]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split

In [2]:
train_df = pd.read_csv("./train.csv")
test_df = pd.read_csv('./test.csv')

In [3]:
import torch

from transformers import AutoModelForMaskedLM, AutoModelForSequenceClassification, AutoTokenizer


tokenizer = AutoTokenizer.from_pretrained(
   "vinai/bertweet-base",
   revision="c90e62314c9035a85cd450ebaa4c986172aa57f1",
)
model = AutoModelForSequenceClassification.from_pretrained(
    "vinai/bertweet-base",
    revision="c90e62314c9035a85cd450ebaa4c986172aa57f1",
    use_safetensors=True,
    device_map="auto"
)
inputs = tokenizer("Plants create <mask> through a process known as photosynthesis.", return_tensors="pt").to(model.device)

with torch.no_grad():
    outputs = model(**inputs)
    predictions = outputs.logits

masked_index = torch.where(inputs['input_ids'] == tokenizer.mask_token_id)[1]
predicted_token_id = predictions[0, masked_index].argmax(dim=-1)
predicted_token = tokenizer.decode(predicted_token_id)

print(f"The predicted token is: {predicted_token}")

/Users/haideeyan/sources/nlp_disaster_improved/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
emoji is not installed, thus not converting emoticons or emojis into text. Install emoji: pip3 install emoji==0.6.0
Some weights of RobertaForSequenceClassification were not initialized from the model checkpoint at vinai/bertweet-base and are newly initialized: ['classifier.dense.bias', 'classifier.dense.weight', 'classifier.out_proj.bias', 'classifier.out_proj.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


The predicted token is: <s>


In [4]:
train_part, valid_part = train_test_split(
    train_df,
    test_size=0.15,
    random_state=42,
    stratify=train_df["target"],
)

print(train_part.shape, valid_part.shape)
train_df.head()

(6471, 5) (1142, 5)


,id,keyword,location,text,target
0,1,NaN,NaN,Our Deeds are the Reason of this #earthquake M...,1
1,4,NaN,NaN,Forest fire near La Ronge Sask. Canada,1
2,5,NaN,NaN,All residents asked to 'shelter in place' are ...,1
3,6,NaN,NaN,"13,000 people receive #wildfires evacuation or...",1
4,7,NaN,NaN,Just got sent this photo from Ruby #Alaska as ...,1


In [5]:
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
)

revision = "c90e62314c9035a85cd450ebaa4c986172aa57f1"

tokenizer = AutoTokenizer.from_pretrained(
    "vinai/bertweet-base",
    revision=revision,
)

model = AutoModelForSequenceClassification.from_pretrained(
    "vinai/bertweet-base",
    revision=revision,
    use_safetensors=True,
    num_labels=2,
)

# A new classification layer is expected and will be trained below.

emoji is not installed, thus not converting emoticons or emojis into text. Install emoji: pip3 install emoji==0.6.0
Some weights of RobertaForSequenceClassification were not initialized from the model checkpoint at vinai/bertweet-base and are newly initialized: ['classifier.dense.bias', 'classifier.dense.weight', 'classifier.out_proj.bias', 'classifier.out_proj.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


In [6]:
import torch
from torch.utils.data import Dataset

class TweetDataset(Dataset):
    def __init__(self, texts, labels=None):
        self.encodings = tokenizer(
            texts.tolist(),
            truncation=True,
            max_length=128,
        )
        self.labels = labels.tolist() if labels is not None else None

    def __len__(self):
        return len(self.encodings["input_ids"])

    def __getitem__(self, index):
        item = {
            key: torch.tensor(values[index])
            for key, values in self.encodings.items()
        }

        if self.labels is not None:
            item["labels"] = torch.tensor(
                self.labels[index],
                dtype=torch.long,
            )

        return item


train_dataset = TweetDataset(
    train_part["text"],
    train_part["target"],
)

valid_dataset = TweetDataset(
    valid_part["text"],
    valid_part["target"],
)

test_dataset = TweetDataset(test_df["text"])

In [7]:
import numpy as np
from sklearn.metrics import accuracy_score, f1_score
from transformers import (
    DataCollatorWithPadding,
    TrainingArguments,
    Trainer,
)

data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

def compute_metrics(eval_prediction):
    logits, labels = eval_prediction
    predictions = np.argmax(logits, axis=-1)

    return {
        "accuracy": accuracy_score(labels, predictions),
        "f1": f1_score(labels, predictions),
    }


training_args = TrainingArguments(
    output_dir="./bertweet-disaster-model",
    learning_rate=2e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=32,
    num_train_epochs=3,
    weight_decay=0.01,
    warmup_ratio=0.1,
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="f1",
    greater_is_better=True,
    save_safetensors=True,
    report_to="none",
    seed=42,
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=valid_dataset,
    data_collator=data_collator,
    compute_metrics=compute_metrics,
)

In [8]:
trainer.train()
trainer.evaluate()

trainer.save_model("./bertweet-disaster-final")
tokenizer.save_pretrained("./bertweet-disaster-final")

predictions = trainer.predict(test_dataset)
predicted_targets = np.argmax(predictions.predictions, axis=-1)

submission = pd.DataFrame({
    "id": test_df["id"],
    "target": predicted_targets,
})

submission.to_csv("submission.csv", index=False)
submission.head()

Epoch,Training Loss,Validation Loss,Accuracy,F1
1,No log,0.399169,0.836252,0.808205
2,0.462600,0.416387,0.845009,0.827317
3,0.333300,0.411417,0.853765,0.825131


,id,target
0,0,1
1,2,1
2,3,1
3,9,1
4,11,1
